In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import subprocess
import sys

packages = [
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "httpx==0.27.*",
    "openai==1.54.*"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages
    ],
    check=True
)

print("INSTALLATION COMPLETE")
print("RESTART SESSION NOW")

INSTALLATION COMPLETE
RESTART SESSION NOW


In [3]:
import os
import subprocess
import sys

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SERVER_LOG = "/kaggle/working/load_shedding_server.log"

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(SERVER_LOG, "w")

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_ID,
        "--dtype",
        "half",
        "--max-model-len",
        "4096",
        "--gpu-memory-utilization",
        "0.85",
        "--disable-frontend-multiprocessing",
        "--port",
        "8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=environment
)

print("SERVER LAUNCHED")
print("PID:", server.pid)
print("LOG:", SERVER_LOG)

SERVER LAUNCHED
PID: 164
LOG: /kaggle/working/load_shedding_server.log


In [4]:
import time
import httpx

MODELS_URL = "http://localhost:8000/v1/models"

print("Waiting for the vLLM server...")

for attempt in range(80):
    try:
        response = httpx.get(
            MODELS_URL,
            timeout=10.0
        )

        if response.status_code == 200:
            print("SERVER HEALTHY:", response.status_code)
            print("Model:", response.json()["data"][0]["id"])
            break

    except Exception:
        pass

    if server.poll() is not None:
        print("SERVER FAILED")

        with open(
            "/kaggle/working/load_shedding_server.log",
            "r",
            errors="replace"
        ) as file:
            print(file.read()[-5000:])

        break

    time.sleep(3)

else:
    print("SERVER TIMEOUT")

    with open(
        "/kaggle/working/load_shedding_server.log",
        "r",
        errors="replace"
    ) as file:
        print(file.read()[-5000:])

Waiting for the vLLM server...
SERVER HEALTHY: 200
Model: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
import asyncio
import time
import httpx

BASE_URL = "http://localhost:8000/v1"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

async def send_one(
    client,
    base_url,
    model,
    prompt,
    max_tokens=128
):
    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": max_tokens,
        "temperature": 0.0
    }

    start = time.perf_counter()

    try:
        response = await client.post(
            f"{base_url}/chat/completions",
            json=payload
        )

        response.raise_for_status()

        return {
            "ok": True,
            "latency_s": time.perf_counter() - start
        }

    except Exception as error:
        return {
            "ok": False,
            "latency_s": time.perf_counter() - start,
            "error": str(error)
        }


def p95(latencies):
    sorted_latencies = sorted(latencies)

    index = max(
        0,
        int(len(sorted_latencies) * 0.95) - 1
    )

    return sorted_latencies[index]

print("HELPER FUNCTIONS READY")

HELPER FUNCTIONS READY


In [7]:
async def naive_burst(
    base_url,
    model,
    prompt,
    n=50,
    max_tokens=128
):
    async with httpx.AsyncClient(
        timeout=120.0
    ) as client:
        results = await asyncio.gather(
            *[
                send_one(
                    client,
                    base_url,
                    model,
                    prompt,
                    max_tokens
                )
                for _ in range(n)
            ]
        )

    latencies = [
        result["latency_s"]
        for result in results
        if result["ok"]
    ]

    return {
        "n_sent": n,
        "n_ok": len(latencies),
        "p95_s": (
            round(p95(latencies), 3)
            if latencies
            else None
        ),
        "mean_s": (
            round(
                sum(latencies) / len(latencies),
                3
            )
            if latencies
            else None
        )
    }


prompt = (
    "In two sentences, explain "
    "what a load balancer does."
)

naive_result = await naive_burst(
    base_url=BASE_URL,
    model=MODEL_ID,
    prompt=prompt,
    n=50
)

print("naive (unbounded):", naive_result)

naive (unbounded): {'n_sent': 50, 'n_ok': 50, 'p95_s': 0.979, 'mean_s': 0.973}


In [8]:
class LoadShedder:

    def __init__(self, max_in_flight: int):
        self.sem = asyncio.Semaphore(max_in_flight)

    async def try_admit(self):
        acquired = (
            self.sem.locked() is False
            and self.sem._value > 0
        )

        if acquired:
            await self.sem.acquire()

        return acquired

    def release(self):
        self.sem.release()


async def send_with_shedding(
    client,
    shedder,
    base_url,
    model,
    prompt,
    max_tokens=128
):
    admitted = await shedder.try_admit()

    if not admitted:
        return {
            "ok": False,
            "shed": True,
            "latency_s": 0.0
        }

    start = time.perf_counter()

    try:
        response = await client.post(
            f"{base_url}/chat/completions",
            json={
                "model": model,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "max_tokens": max_tokens,
                "temperature": 0.0
            }
        )

        response.raise_for_status()

        return {
            "ok": True,
            "shed": False,
            "latency_s": time.perf_counter() - start
        }

    except Exception as error:
        return {
            "ok": False,
            "shed": False,
            "latency_s": time.perf_counter() - start,
            "error": str(error)
        }

    finally:
        shedder.release()


async def shedded_burst(
    base_url,
    model,
    prompt,
    n=50,
    cap=8,
    max_tokens=128
):
    shedder = LoadShedder(cap)

    async with httpx.AsyncClient(
        timeout=120.0
    ) as client:
        results = await asyncio.gather(
            *[
                send_with_shedding(
                    client,
                    shedder,
                    base_url,
                    model,
                    prompt,
                    max_tokens
                )
                for _ in range(n)
            ]
        )

    accepted = [
        result
        for result in results
        if result["ok"]
    ]

    shed = [
        result
        for result in results
        if result.get("shed")
    ]

    latencies = [
        result["latency_s"]
        for result in accepted
    ]

    return {
        "n_sent": n,
        "cap": cap,
        "n_accepted": len(accepted),
        "n_shed": len(shed),
        "accepted_p95_s": (
            round(p95(latencies), 3)
            if latencies
            else None
        )
    }


shedded_result = await shedded_burst(
    base_url=BASE_URL,
    model=MODEL_ID,
    prompt=prompt,
    n=50,
    cap=8
)

print("shedded (cap=8):", shedded_result)

shedded (cap=8): {'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.458}


In [9]:
sweep = []

for n in (8, 16, 32, 50):
    result = await shedded_burst(
        base_url=BASE_URL,
        model=MODEL_ID,
        prompt=prompt,
        n=n,
        cap=8
    )

    sweep.append(result)
    print(result)

print("LOAD SHEDDING SWEEP COMPLETE")

{'n_sent': 8, 'cap': 8, 'n_accepted': 8, 'n_shed': 0, 'accepted_p95_s': 0.441}
{'n_sent': 16, 'cap': 8, 'n_accepted': 8, 'n_shed': 8, 'accepted_p95_s': 0.41}
{'n_sent': 32, 'cap': 8, 'n_accepted': 8, 'n_shed': 24, 'accepted_p95_s': 0.404}
{'n_sent': 50, 'cap': 8, 'n_accepted': 8, 'n_shed': 42, 'accepted_p95_s': 0.405}
LOAD SHEDDING SWEEP COMPLETE


In [11]:
import json

REPORT_PATH = "/kaggle/working/shedding_report.json"

report = {
    "naive_unbounded_n50": naive_result,
    "shedded_cap8_n50": shedded_result,
    "shedded_sweep": sweep
}

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

print(json.dumps(report, indent=2))
print("SHEDDING REPORT CREATED")

{
  "naive_unbounded_n50": {
    "n_sent": 50,
    "n_ok": 50,
    "p95_s": 0.979,
    "mean_s": 0.973
  },
  "shedded_cap8_n50": {
    "n_sent": 50,
    "cap": 8,
    "n_accepted": 8,
    "n_shed": 42,
    "accepted_p95_s": 0.458
  },
  "shedded_sweep": [
    {
      "n_sent": 8,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 0,
      "accepted_p95_s": 0.441
    },
    {
      "n_sent": 16,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 8,
      "accepted_p95_s": 0.41
    },
    {
      "n_sent": 32,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 24,
      "accepted_p95_s": 0.404
    },
    {
      "n_sent": 50,
      "cap": 8,
      "n_accepted": 8,
      "n_shed": 42,
      "accepted_p95_s": 0.405
    }
  ]
}
SHEDDING REPORT CREATED


In [12]:
import json

REPORT_PATH = "/kaggle/working/shedding_report.json"

with open(REPORT_PATH, "r", encoding="utf-8") as file:
    report = json.load(file)

naive = report["naive_unbounded_n50"]
shedded = report["shedded_cap8_n50"]
sweep_results = report["shedded_sweep"]

assert naive["n_sent"] == 50
assert naive["n_ok"] >= 45
assert naive["p95_s"] > 0

assert shedded["n_sent"] == 50
assert shedded["cap"] == 8
assert shedded["n_accepted"] >= 8
assert shedded["n_shed"] >= 20
assert shedded["n_accepted"] + shedded["n_shed"] <= 50
assert shedded["accepted_p95_s"] > 0
assert shedded["accepted_p95_s"] < naive["p95_s"] * 0.8

assert [item["n_sent"] for item in sweep_results] == [8, 16, 32, 50]
assert all(item["cap"] == 8 for item in sweep_results)

shed_counts = [
    item["n_shed"]
    for item in sweep_results
]

accepted_p95_values = [
    item["accepted_p95_s"]
    for item in sweep_results
]

assert shed_counts[0] <= 1
assert all(
    current <= following
    for current, following in zip(
        shed_counts,
        shed_counts[1:]
    )
)
assert shed_counts[-1] >= 10
assert max(accepted_p95_values) <= min(accepted_p95_values) * 2.5

print("invariants hold: shedding happened, accepted p95 protected, cap flat")
print("GREEN CHECK: PASS")

invariants hold: shedding happened, accepted p95 protected, cap flat
GREEN CHECK: PASS


In [13]:
import base64
from IPython.display import HTML, display

file_path = "/kaggle/working/shedding_report.json"

with open(file_path, "rb") as file:
    encoded = base64.b64encode(file.read()).decode()

download_link = f"""
<a download="shedding_report.json"
   href="data:application/json;base64,{encoded}">
   Download shedding_report.json
</a>
"""

display(HTML(download_link))